# Automated image registration

This notebook demonstrates the registration of images from H&E, IHC or IF stainings that were performed on the same slide as the Xenium In Situ measurements. It is assumed that the images which are about to be registered, contain the same tissue as the spatial transcriptomics data. 


In [1]:
## The following code ensures that all functions and init files are reloaded before executions.
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from insitupy import CACHE
from insitupy.io import read_xenium
from insitupy.tools import register_images

c:\Users\ge37voy\AppData\Local\miniforge3\envs\test_rev\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Xenium data into `InSituData` object

Now the Xenium data can be parsed by providing the data path to `InSituData` using the `read_xenium` function or directly using the downloading function.

### Load the dataset directly from the downloading function...

In [3]:
from insitupy.datasets import xenium_human_breast_cancer

In [4]:
xd = xenium_human_breast_cancer()

2026-03-17 13:38:18 | [INFO] This dataset exists already. Download is skipped. To force download set `overwrite=True`.
2026-03-17 13:38:18 | [INFO] Image exists. Checking md5sum...
2026-03-17 13:38:21 | [INFO] The md5sum matches. Download is skipped. To force download set `overwrite=True`.
2026-03-17 13:38:21 | [INFO] Image exists. Checking md5sum...
2026-03-17 13:38:21 | [INFO] The md5sum matches. Download is skipped. To force download set `overwrite=True`.
2026-03-17 13:38:21 | [INFO] Corresponding image data can be found in C:\Users\ge37voy\.cache\InSituPy\demo_datasets\xenium_hbreastcancer\unregistered_images
2026-03-17 13:38:21 | [INFO] For this dataset following images are available:
2026-03-17 13:38:21 | [INFO] slide_id__hbreastcancer__HE__histo.ome.tiff
2026-03-17 13:38:21 | [INFO] slide_id__hbreastcancer__CD20_HER2_DAPI__IF.ome.tiff
2026-03-17 13:38:21 | [INFO] Reading Xenium data with InSituPy backend...
2026-03-17 13:38:21 | [INFO] Loading cells...
2026-03-17 13:38:22 | [INF

### ... or use the `read_xenium` function and the path to the Xenium data directory if the dataset has already been downloaded

In [5]:
xd = read_xenium(CACHE / "demo_datasets/xenium_hbreastcancer/output-XETG00000__slide_id__hbreastcancer")

2026-03-17 13:39:10 | [INFO] Reading Xenium data with InSituPy backend...
2026-03-17 13:39:10 | [INFO] Loading cells...
2026-03-17 13:39:11 | [INFO] Loading images...
2026-03-17 13:39:11 | [INFO] Loading transcripts...


In [6]:
xd

InSituData
Method:		Xenium
Slide ID:	0001879
Sample ID:	Replicate 1
Path:		C:\Users\ge37voy\.cache\InSituPy\demo_datasets\xenium_hbreastcancer\output-XETG00000__slide_id__hbreastcancer

    ➤ images
       'nuclei':   (25778, 35416)
    ➤ cells
       MultiCellData with main layer 'main'
           table
               AnnData object with n_obs × n_vars = 167780 × 313
               obs: 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area'
               var: 'gene_ids', 'feature_types', 'genome'
               obsm: 'spatial'
           boundaries
               BoundariesData object with 2 entries:
                   cells
                   nuclei
    ➤ transcripts
       DataFrame with shape <dask_expr.expr.Scalar: expr=ReadParquetFSSpec(de5d330).size() // 8, dtype=int64> x 8

### Prepare the paths to the unregistered images

Here the unregistered images were downloaded by the `human_breast_cancer` downloading function and saved in a folder `unregistered_images`.

In [7]:
# prepare paths
if_to_be_registered = CACHE / "demo_datasets/xenium_hbreastcancer" / "unregistered_images/slide_id__hbreastcancer__CD20_HER2_DAPI__IF.ome.tif"
he_to_be_registered = CACHE / "demo_datasets/xenium_hbreastcancer" / "unregistered_images/slide_id__hbreastcancer__HE__histo.ome.tif"

### Automated Registration of Images

**Overview:**
_Xenium In Situ_ is a non-destructive method that allows staining and imaging of tissue after in situ sequencing analysis. Because this is performed outside the _Xenium_ instrument, the acquired images must be registered back to the Xenium reference. `InSituPy` provides an automatic registration pipeline based on the [Scale-Invariant Feature Transform (SIFT) algorithm](https://link.springer.com/article/10.1023/B:VISI.0000029664.99615.94).

**Process:**
1. **Feature Detection and Transformation:**
   - SIFT detects common features between the template (_Xenium_ nuclei/DAPI image) and the acquired image.
   - Feature matches are used to compute an affine transformation matrix.
   - The transformation matrix is then applied to register the image to the template.

<left><img src="../../demo_data/demo_screenshots/common_features.jpg" width="800"/></left>

*Common features extracted by the SIFT algorithm*

2. **Preprocessing Steps:**
   - **Histological images (H&E or IHC):**
     - Input is expected to be RGB.
     - Color deconvolution extracts the hematoxylin-rich nuclei signal used for registration against Xenium nuclei/DAPI.
   - **Immunofluorescence (IF) images:**
     - Input is expected to be multi-channel.
     - One channel must contain a nuclei stain (for example, DAPI).
     - This channel is used for SIFT feature detection and transformation estimation.
     - The same transformation is applied to the remaining channels.

### Cropping of Images from Whole Slide Images

**Workflow:**
In a Xenium In Situ workflow, one slide often contains multiple tissue sections. Spatial transcriptomics outputs are already separated per sample, while histology files often contain the full slide. To extract per-section histology images, two workflows are recommended:

1. **QuPath annotation:**
   - Annotate and name individual tissue sections in QuPath.
   - Use `InSituPy/scripts/export_annotations_OME-TIFF.groovy` to export OME-TIFF crops.

2. **Napari-based approach:**
   - Demonstrated in `XX_InSituPy_extract_individual_images.ipynb`.

### Input Files

**Formats:**
- `.tif` and `.ome.tif` are supported.
- **IF images:**
  - Multi-channel images are expected.
  - Provide channel names via `channel_names`.
  - Provide the nuclei channel via `channel_name_for_registration` (for example, DAPI).
- **HE images:**
  - RGB images are expected.
  - Cropping should preserve correct channel layout/metadata.

### Output Generated by the Registration Pipeline

1. **Registered images**
   - If `save_registered_images=True`, registered images are saved as `.ome.tif` in `registered_images`.
   - Naming convention: `slide_id__sample_id__name__registered.ome.tif`.

2. **QC outputs** (saved in `registered_images/registration_qc`)
   - QC files are written for successful runs when `debug=True`.
   - A failure QC snapshot is also written when feature matching fails.
   - Transformation matrix: `slide_id__sample_id__name__transform.csv`.
   - Match overview image: `slide_id__sample_id__name__matches_overview.png`.
   - Match detail image (top matches): `slide_id__sample_id__name__matches_detail.png`.

**Directory Structure:**
```
./demo_dataset
├───output-XETG00000__slide_id__sample_id
├───registered_images
│   │   slide_id__sample_id__name__registered.ome.tif
│   ├───registration_qc
│   │       slide_id__sample_id__name__transform.csv
│   │       slide_id__sample_id__name__matches_overview.png
│   │       slide_id__sample_id__name__matches_detail.png
└───unregistered_images
```

## Registration of IF images

In [8]:
register_images(
    data=xd,
    image_path=if_to_be_registered,
    channel_names=['CD20', 'HER2', 'DAPI'],
    channel_name_for_registration="DAPI",
    template_image_name="nuclei",
    save_registered_images=True,
    rank_matches_for_qc=True,
    debug=True,  # set False for faster runs without routine QC exports
    )

2026-03-17 13:39:24 | [INFO]   ├── Loading images
2026-03-17 13:39:24 | [INFO] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2026-03-17 13:39:24 | [INFO] Registration: 0001879__Replicate 1 ── CD20, HER2, DAPI (IF)
2026-03-17 13:39:24 | [INFO] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2026-03-17 13:39:24 | [INFO]   │     Image:    (3, 9777, 14239)
2026-03-17 13:39:24 | [INFO]   │     Template: (25778, 35416)
2026-03-17 13:39:24 | [INFO]   ├── Selecting nuclei channel (index: 2)
2026-03-17 13:39:25 | [INFO]   ├── Loading images into memory
2026-03-17 13:39:39 | [INFO]   ├── Scaling (full image prep)
2026-03-17 13:39:40 | [INFO]   │     Image:    (3, 9777, 14239) → (3, 3314, 4827)
2026-03-17 13:39:40 | [INFO]   │     Template: (25778, 35416) → (3412, 4688)
2026-03-17 13:39:40 | [INFO]   ├── Scaling (registration channel)
2026-03-17 13:39:40 | [INFO]   │     Image:    (9777, 14239) → (3314, 4827)
2026-03-17 13:39:40 

In [9]:
xd

InSituData
Method:		Xenium
Slide ID:	0001879
Sample ID:	Replicate 1
Path:		C:\Users\ge37voy\.cache\InSituPy\demo_datasets\xenium_hbreastcancer\output-XETG00000__slide_id__hbreastcancer

    ➤ images
       'nuclei':   (25778, 35416)
       'CD20':     (25778, 35416)
       'HER2':     (25778, 35416)
    ➤ cells
       MultiCellData with main layer 'main'
           table
               AnnData object with n_obs × n_vars = 167780 × 313
               obs: 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area'
               var: 'gene_ids', 'feature_types', 'genome'
               obsm: 'spatial'
           boundaries
               BoundariesData object with 2 entries:
                   cells
                   nuclei
    ➤ transcripts
       DataFrame with shape <dask_expr.expr.Scalar: expr=ReadParquetFSSpec(de5d330).size() // 8, dtype=int64> x 8

### Adding individual channels

The example above shows how to deal with multi-channel IF images where one channel contains nuclei information and can therefore be used to register the images automatically. In alternative scenarios, one might have single-channel images and wants to add those to the `InSituData`. In such cases one needs the transformation matrix to align the image with the existing `InSituData` and then can add the image using `ImageData`'s `add_image` function as shown in the example below.

In [10]:
# read the multi-channel image and extract an individual channel for demonstration
from insitupy.images.io import read_image
img_pyramid, ome_meta, axes, pixel_size = read_image(if_to_be_registered)
img = img_pyramid[0] # select highest resolution
img_ch = img[0, :, :]

In [11]:
img_ch

dask.array<getitem, shape=(9777, 14239), dtype=uint8, chunksize=(1024, 1024), chunktype=numpy.ndarray>

In [12]:
# read the transformation matrix from the previous registration
transformation_matrix = CACHE / "demo_datasets/xenium_hbreastcancer/registered_images/registration_qc/0001879__Replicate 1__DAPI__transform.csv"

Add the image to the `ImageData` object and specify the `transformation matrix`:

In [13]:
xd.images.add_image(
    image=img_ch,
    channel_names="CD20_2",
    axes="YX",
    pixel_size=pixel_size,
    transformation_matrix=transformation_matrix,
    reference_image="nuclei",
    overwrite=True
)

2026-03-17 13:41:32 | [INFO] Applying transformation to image 'CD20_2'...
2026-03-17 13:41:32 | [INFO] Using reference image 'nuclei' (pixel size: 0.2125 µm/pixel, shape: 25778x35416 pixels = 5477.8x7525.9 µm)
2026-03-17 13:41:32 | [INFO] Converted transformation matrix from pixel coordinates (reference: 0.2125 µm/pixel) to physical coordinates.
2026-03-17 13:41:32 | [INFO] Applying transformation matrix (in physical coordinates):
[[-3.47580326e+00  2.48591853e-02  9.02645355e+03]
 [-2.45060970e-02 -3.47891712e+00  6.88766822e+03]]
2026-03-17 13:41:33 | [INFO] Transforming image 'CD20_2' with shape (9777, 14239) -> output size (35416, 25778)
2026-03-17 13:41:34 | [INFO] Transformed image 'CD20_2'
2026-03-17 13:41:34 | [INFO] Transformed 1 images.


The single image has been added to the `ImageData` object and results can be visualized using `xd.show()`.

In [14]:
xd.images

'nuclei':   (25778, 35416)
'CD20':     (25778, 35416)
'HER2':     (25778, 35416)
'CD20_2':   (25778, 35416)

In [15]:
xd.show()

2026-03-17 13:42:07 | [INFO] Extracting unique gene names from Dask DataFrame...
2026-03-17 13:42:13 | [INFO] Found 541 unique genes


Remove the additional CD20 image again.

In [16]:
del xd.images['CD20_2']

In [17]:
xd.images

'nuclei':   (25778, 35416)
'CD20':     (25778, 35416)
'HER2':     (25778, 35416)

## Registration of H&E images

In [18]:
register_images(
    data=xd,
    image_path=he_to_be_registered,
    channel_names='HE',
    template_image_name="nuclei",
    save_registered_images=True,
    debug=True
    )

C:\Users\ge37voy\AppData\Local\Temp\ipykernel_29756\1269042246.py:1: DeprecationWarning: `image_to_be_registered` is deprecated and will be removed in a future release. Use `image_path` instead.
  register_images(


2026-03-17 13:42:39 | [INFO]   ├── Loading images
2026-03-17 13:42:39 | [INFO] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2026-03-17 13:42:39 | [INFO] Registration: 0001879__Replicate 1 ── HE (histo)
2026-03-17 13:42:39 | [INFO] ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2026-03-17 13:42:39 | [INFO]   │     Image:    (24241, 30786, 3)
2026-03-17 13:42:39 | [INFO]   │     Template: (25778, 35416)
2026-03-17 13:42:39 | [INFO]   ├── Color deconvolution (scale factor: 0.2)
2026-03-17 13:43:48 | [INFO]   ├── Loading images into memory
2026-03-17 13:44:09 | [INFO]   ├── Scaling (full image prep)
2026-03-17 13:44:09 | [INFO]   │     Image:    (24241, 30786, 3) → (3548, 4507, 3)
2026-03-17 13:44:09 | [INFO]   │     Template: (25778, 35416) → (3412, 4688)
2026-03-17 13:44:09 | [INFO]   ├── Scaling (registration channel)
2026-03-17 13:44:09 | [INFO]   │     Image:    (24240, 30785) → (3548, 4507)
2026-03-17 13:44:09 | [I

In [19]:
xd

InSituData
Method:		Xenium
Slide ID:	0001879
Sample ID:	Replicate 1
Path:		C:\Users\ge37voy\.cache\InSituPy\demo_datasets\xenium_hbreastcancer\output-XETG00000__slide_id__hbreastcancer

    ➤ images
       'nuclei':   (25778, 35416)
       'CD20':     (25778, 35416)
       'HER2':     (25778, 35416)
       'HE':       (25778, 35416, 3)
    ➤ cells
       MultiCellData with main layer 'main'
           table
               AnnData object with n_obs × n_vars = 167780 × 313
               obs: 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area'
               var: 'gene_ids', 'feature_types', 'genome'
               obsm: 'spatial'
           boundaries
               BoundariesData object with 2 entries:
                   cells
                   nuclei
    ➤ transcripts
       DataFrame with shape <dask_expr.expr.Scalar: expr=ReadParquetFSSpec(de5d330).size() // 8, dtype=int64> x 8

Visualize the aligned images.

In [47]:
xd.show()

2026-03-03 11:53:25 | [INFO] Extracting unique gene names from Dask DataFrame...
2026-03-03 11:53:30 | [INFO] Found 541 unique genes


## Working with an `InSituPy` project

To allow a simple and structured saving workflow, `InSituPy` provides two saving functions:
- `saveas()`
- `save()`


### Save as `InSituPy` project

In [ ]:
insitupy_project = Path(CACHE / "out/demo_insitupy_project")

In [21]:
xd.saveas(insitupy_project, overwrite=True)

2026-03-17 13:47:12 | [INFO] Saving data to C:\Users\ge37voy\.cache\InSituPy\out\demo_insitupy_project2


... storing 'feature_types' as categorical
... storing 'genome' as categorical


2026-03-17 13:49:28 | [INFO] Saved.


### Save `InSituPy` project with downscaled image data

Since the image data is very large and not required during most of the trancriptomic analysis, we can downscale the image data to save disk space.

In [22]:
insitupy_project_downscaled = Path(CACHE / "out/demo_insitupy_project_downscaled")
xd.saveas(
    insitupy_project_downscaled,
    images_max_resolution=1, # in µm/pixel
    overwrite=True
    )

2026-03-17 13:49:29 | [INFO] Saving data to C:\Users\ge37voy\.cache\InSituPy\out\demo_insitupy_project_downscaled
2026-03-17 13:49:29 | [INFO] Downscale image to 1 µm per pixel by factor 4.705882352941177
2026-03-17 13:49:40 | [INFO] Downscale image to 1 µm per pixel by factor 4.705882352941177
2026-03-17 13:49:42 | [INFO] Downscale image to 1 µm per pixel by factor 4.705882352941177
2026-03-17 13:49:43 | [INFO] Downscale image to 1 µm per pixel by factor 4.705882352941177
2026-03-17 13:50:07 | [INFO] Saved.


### Reload from `InSituPy` project

From the `InSituPy` project we can now load only the modalities that we need for later analyses. Due to an optimized file structure using `zarr` and `dask`, this makes loading and visualization of the data more efficient compared to doing this directly from the xenium data bundle.

In [23]:
from insitupy import InSituData

In [24]:
xd = InSituData.read(insitupy_project)
xd_ds = InSituData.read(insitupy_project_downscaled)

In [25]:
xd

InSituData
Method:		Xenium
Slide ID:	0001879
Sample ID:	Replicate 1
Path:		C:\Users\ge37voy\.cache\InSituPy\out\demo_insitupy_project2


No modalities loaded.

In [26]:
xd_ds

InSituData
Method:		Xenium
Slide ID:	0001879
Sample ID:	Replicate 1
Path:		C:\Users\ge37voy\.cache\InSituPy\out\demo_insitupy_project_downscaled


No modalities loaded.

### Load all required modalities

Next, we have to make sure that all data modalities that are required for the subsequent analyses are loaded. In our case it is the cellular data and the image data. If a modality is missing, one can load it with `.load_{modality}`.

Load selected modalities of down-scaled data:

In [27]:
xd_ds.load_cells()
xd_ds.load_images()

In [28]:
xd_ds

InSituData
Method:		Xenium
Slide ID:	0001879
Sample ID:	Replicate 1
Path:		C:\Users\ge37voy\.cache\InSituPy\out\demo_insitupy_project_downscaled

    ➤ images
       'CD20':     (5477, 7525)
       'HE':       (5477, 7525, 3)
       'HER2':     (5477, 7525)
       'nuclei':   (5477, 7525)
    ➤ cells
       MultiCellData with main layer 'main'
           table
               AnnData object with n_obs × n_vars = 167780 × 313
               obs: 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area'
               var: 'gene_ids', 'feature_types', 'genome'
               obsm: 'spatial'
           boundaries
               BoundariesData object with 2 entries:
                   cells
                   nuclei

In [29]:
xd_ds.show()

Load all modalities but the `transcripts` of the full scale data:

In [30]:
xd.load_all(skip="transcripts")

In [31]:
xd

InSituData
Method:		Xenium
Slide ID:	0001879
Sample ID:	Replicate 1
Path:		C:\Users\ge37voy\.cache\InSituPy\out\demo_insitupy_project2

    ➤ images
       'CD20':     (25778, 35416)
       'HE':       (25778, 35416, 3)
       'HER2':     (25778, 35416)
       'nuclei':   (25778, 35416)
    ➤ cells
       MultiCellData with main layer 'main'
           table
               AnnData object with n_obs × n_vars = 167780 × 313
               obs: 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area'
               var: 'gene_ids', 'feature_types', 'genome'
               obsm: 'spatial'
           boundaries
               BoundariesData object with 2 entries:
                   cells
                   nuclei

In [32]:
xd.show()